In [1]:
!pip install langchain-community langchain-openai sqlalchemy

In [3]:
import sqlite3

# (可选) 如果你希望通过Python脚本创建数据库和表
def setup_database(db_file="ecommerce.db"):
    conn = sqlite3.connect(db_file)
    cursor = conn.cursor()
    
    # 执行上述SQL脚本
    script = """
    DROP TABLE IF EXISTS customers;
    DROP TABLE IF EXISTS orders;
    CREATE TABLE customers (id INTEGER PRIMARY KEY, name VARCHAR(50) NOT NULL, age INTEGER, city VARCHAR(50));
    INSERT INTO customers (id, name, age, city) VALUES (1, 'Alice', 30, 'New York'), (2, 'Bob', 25, 'Los Angeles'), (3, 'Charlie', 35, 'New York'), (4, 'David', 42, 'Chicago');
    CREATE TABLE orders (order_id INTEGER PRIMARY KEY, customer_id INTEGER, product VARCHAR(50), amount DECIMAL(10, 2), order_date DATE, FOREIGN KEY (customer_id) REFERENCES customers(id));
    INSERT INTO orders (order_id, customer_id, product, amount, order_date) VALUES (101, 1, 'Laptop', 1200.00, '2025-08-01'), (102, 2, 'Mouse', 25.00, '2025-08-05'), (103, 1, 'Keyboard', 75.50, '2025-08-10'), (104, 3, 'Monitor', 300.00, '2025-08-12'), (105, 1, 'Webcam', 50.00, '2025-08-15'), (106, 4, 'USB Hub', 15.75, '2025-08-20');
    """
    cursor.executescript(script)
    conn.commit()
    conn.close()
    print(f"数据库 '{db_file}' 已成功创建和初始化。")

# 运行设置函数
setup_database()

# 使用 LangChain 连接数据库
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///ecommerce.db")

# 验证连接和基本信息
print(f"数据库方言 (Dialect): {db.dialect}")
print(f"可用的表名 (Usable tables): {db.get_usable_table_names()}")

数据库 'ecommerce.db' 已成功创建和初始化。
数据库方言 (Dialect): sqlite
可用的表名 (Usable tables): ['customers', 'orders']


In [4]:
from langchain_core.tools import tool
import json

# 确保 db 对象已在前面步骤中创建
# db = SQLDatabase.from_uri("sqlite:///ecommerce.db")

@tool
def list_tables() -> str:
    """
    返回一个包含数据库中所有可用表名的列表。
    当你需要知道数据库中有哪些表时，应该首先调用此工具。
    输入参数为空。
    """
    table_names = db.get_usable_table_names()
    return json.dumps({"table_names": table_names})

@tool
def get_table_schema(table_names: str) -> str:
    """
    根据一个用逗号分隔的表名字符串，返回这些表的详细结构信息(schema)。
    在你已经知道表名但需要了解其具体列名、数据类型和主外键关系时使用。
    例如，输入 'customers,orders'。
    """
    # LangChain 的 get_table_info 期望一个列表作为参数
    tables = [name.strip() for name in table_names.split(',')]
    schema_info = db.get_table_info(table_names=tables)
    return schema_info

# 将手动创建的工具放入一个列表，以便后续提供给 Agent
manual_tools = [list_tables, get_table_schema]

# 我们可以检查一下工具的结构
print("--- list_tables 工具信息 ---")
print(f"名称: {list_tables.name}")
print(f"描述: {list_tables.description}")
print(f"参数 Schema: {list_tables.args}")

print("\n--- get_table_schema 工具信息 ---")
print(f"名称: {get_table_schema.name}")
print(f"描述: {get_table_schema.description}")
print(f"参数 Schema: {get_table_schema.args}")

--- list_tables 工具信息 ---
名称: list_tables
描述: 返回一个包含数据库中所有可用表名的列表。
当你需要知道数据库中有哪些表时，应该首先调用此工具。
输入参数为空。
参数 Schema: {}

--- get_table_schema 工具信息 ---
名称: get_table_schema
描述: 根据一个用逗号分隔的表名字符串，返回这些表的详细结构信息(schema)。
在你已经知道表名但需要了解其具体列名、数据类型和主外键关系时使用。
例如，输入 'customers,orders'。
参数 Schema: {'table_names': {'title': 'Table Names', 'type': 'string'}}


In [5]:
import os

os.environ["OPENAI_API_KEY"] = "sk-6**********************"
os.environ["OPENAI_BASE_URL"] = "https://dashscope.aliyuncs.com/compatible-mode/v1"
os.environ["OPENAI_MODEL"] = "qwen-turbo"

In [15]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),  # 若走原生 OpenAI，可传 None/不传
    temperature=0.2,                        # 稳定输出
    timeout=1200,                             # 超时保护（秒）
    max_retries=2                           # 简单重试
)

In [16]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_openai import ChatOpenAI
import os

# 实例化 SQLDatabaseToolkit
# 传入 db 和 llm 对象
toolkit = SQLDatabaseToolkit(db=db, llm=llm)

# 获取工具列表
toolkit_tools = toolkit.get_tools()

# 打印出工具包提供了哪些工具
print(f"SQLDatabaseToolkit 提供了 {len(toolkit_tools)} 个工具:")
for tool in toolkit_tools:
    print(f"- 名称: {tool.name}")
    print(f"  描述: {tool.description}\n")

SQLDatabaseToolkit 提供了 4 个工具:
- 名称: sql_db_query
  描述: Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.

- 名称: sql_db_schema
  描述: Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3

- 名称: sql_db_list_tables
  描述: Input is an empty string, output is a comma-separated list of tables in the database.

- 名称: sql_db_query_checker
  描述: Use this tool to double check if your query is correct before executing it. Always use this tool before executing a query with sql_db_query!



In [19]:
from langchain.agents import create_sql_agent
from langchain.agents.agent_types import AgentType

# 使用 create_sql_agent 创建 AgentExecutor
agent_executor = create_sql_agent(
    llm=llm,
    toolkit=toolkit, # 如果使用方案A，这里替换为 tools=manual_tools
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION, # 推荐使用为 Function Calling 优化的 Agent 类型
    verbose=True, # 设为 True，打印 Agent 的完整思考过程，强烈建议在调试时开启
    handle_parsing_errors=True, # 处理解析错误，增强 Agent 的鲁棒性
    prefix="""
    你是一名专业的 SQL 数据分析师。你的任务是根据用户提出的问题，生成语法正确的 SQL 查询语句，
    并执行这些查询来获取答案。
    
    指令:
    1. 在生成 SQL 查询时，除非用户明确指定要获取的示例数量，否则请始终使用 LIMIT 10 来限制返回结果的数量。
    2. 永远不要查询表中的所有列，只选择与问题相关的列。
    3. 请严格使用在表结构信息中看到的列名，不要虚构不存在的列。
    4. 注意每个列属于哪个表，在需要时正确使用 JOIN。
    5. 生成的 SQL 一定要执行，并拿到结果，不要把 SQL 单纯的丢出来
    """
)

In [20]:
question = "查询所有在 New York 的客户以及他们各自的订单总额是多少？"
response = agent_executor.invoke({"input": question})

print("\n--- 最终答案 ---")
print(response["output"])



> Entering new SQL Agent Executor chain...
Action: sql_db_list_tables
Action Input: 
Observcustomers, ordersThought: I need to check the schema of the "customers" and "orders" tables to understand the structure and identify the relevant columns for the query.
Action: sql_db_schema
Action Input: customers, orders

CREATE TABLE customers (
	id INTEGER, 
	name VARCHAR(50) NOT NULL, 
	age INTEGER, 
	city VARCHAR(50), 
	PRIMARY KEY (id)
)

/*
3 rows from customers table:
id	name	age	city
1	Alice	30	New York
2	Bob	25	Los Angeles
3	Charlie	35	New York
*/


CREATE TABLE orders (
	order_id INTEGER, 
	customer_id INTEGER, 
	product VARCHAR(50), 
	amount DECIMAL(10, 2), 
	order_date DATE, 
	PRIMARY KEY (order_id), 
	FOREIGN KEY(customer_id) REFERENCES customers (id)
)

/*
3 rows from orders table:
order_id	customer_id	product	amount	order_date
101	1	Laptop	1200.00	2025-08-01
102	2	Mouse	25.00	2025-08-05
103	1	Keyboard	75.50	2025-08-10
*/Thought: I now know the final answer
Final Answer: 以下是所有在 